# User Progress Model

## Import Libraries

In [4]:
import numpy as np
import pandas as pd
import joblib, pickle

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import (classification_report, mean_absolute_error, 
                            mean_squared_error, r2_score, accuracy_score)

import xgboost as xgb
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder

## Global Configuration

In [5]:
# Define a specific number (Seed) to ensure the code behaves exactly the same way
# every time we run it. This prevents random changes in model scores.
SEED = 26

# Lock the random number generator of NumPy using our Seed.
np.random.seed(SEED)

# Configure Pandas to display ALL columns when printing a dataframe.
# By default, Pandas hides middle columns with "..." if there are too many.
# We disable this limit so we can inspect all our features during debugging.
pd.set_option('display.max_columns', None)

## Preprocessing Data

In [6]:
# Read the CSV file containing user progress history into a Pandas DataFrame.
# '../../datas/' indicates the file is located two folders up in a 'datas' directory.
# Ensure this path matches your actual folder structure.
df_progress = pd.read_csv('../../data/dataset_userprogress.csv')

# [Recommended] Quick Verification
# Print the first 5 rows and the shape (rows, columns) to confirm data loaded successfully.
# This helps catch errors like "FileNotFound" or empty files immediately.
print(f"Data Loaded Successfully. Shape: {df_progress.shape}")

Data Loaded Successfully. Shape: (7800, 33)


In [7]:
# The function tells us:
# 1. How many rows and columns we have.
# 2. The Name and Data Type (Dtype) of each column (int, float, object/text).
# 3. "Non-Null Count": If this number is lower than the total rows, we have missing data!
df_progress.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7800 entries, 0 to 7799
Data columns (total 33 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   User_ID                   7800 non-null   int64  
 1   Age                       7800 non-null   int64  
 2   Gender                    7800 non-null   object 
 3   Height_cm                 7800 non-null   int64  
 4   Initial_Weight_kg         7800 non-null   int64  
 5   Initial_BMI               7800 non-null   float64
 6   BMI_Category_x            7800 non-null   object 
 7   Body_Fat_Category         7800 non-null   float64
 8   Body_Fat_Percentage_x     7800 non-null   float64
 9   Goal                      7800 non-null   object 
 10  Workout_Frequency         7800 non-null   int64  
 11  Average_Duration_Minutes  7800 non-null   int64  
 12  level                     7800 non-null   object 
 13  Badminton                 7800 non-null   int64  
 14  Football

### Encoding Data

In [8]:
# Initialize a dictionary to store our Encoders
# The use of saving encoders:
# - To encode NEW data later (example, a new user registers as "Female", we need to know that = 0).
# - To decode predictions (example, Model predicts class "2", we need to know "2" = "Normal Weight").
le_dict = {}

# Define the map: Original Column Name -> New Numeric Column Name
cols_to_encode = {
    'Gender': 'Gender_Encoded',
    'Goal': 'Goal_Encoded',
    'level': 'level_Encoded',
    'Meal_Frequency': 'Meal_Frequency_Encoded',
    'BMI_Category_x': 'BMI_Category_x_Encoded',     # Input: The user's starting category
    'BMI_Category_y': 'BMI_Category_y_Encoded'      # Target: The category we want to predict
}

# Execution Loop
for col_name, col_encoded in cols_to_encode.items():
    # Create a specific translator for this column
    le = LabelEncoder()
    
    # fit_transform does two jobs:
    # - Fit: Learns the unique words (e.g., ["Beginner", "Intermediate", "Advanced"])
    # - Transform: Converts them to numbers (e.g., [0, 2, 1])
    df_progress[col_encoded] = le.fit_transform(df_progress[col_name])
    
    # Store the translator for future use
    le_dict[col_name] = le

### Splitting Dataset

We can't split data straight out or randomly, since we cant have same user but in train and test set because this will cause data leakage. So we need to divide and decide which set will the user be used for.

In [9]:
# Create a "User Profile" Table
# Problem: If we split random rows, User A's "Week 1" data might be in Train
# and User A's "Week 2" data in Test. The model would just memorize User A.
# Solution: We group by User_ID and take the first row to get their static profile.
user_metadata = df_progress.groupby('User_ID').first()

# Define Inputs for the Splitter
user_ids = user_metadata.index                    # List of unique User IDs (e.g., [1, 2, 3...])
stratify_labels = user_metadata['BMI_Category_y'] # The target we want to balance.

# The use of stratify:
# If we have 100 users and only 5 are "Obese", a random split might put all 5 in Train.
# Stratify forces the split to put ~3 in Train and ~2 in Test, so we can test properly.

# Perform the Split
train_users, test_users = train_test_split(
    user_ids,                # We split the IDs, not the full dataframe
    test_size=0.3,           # 30% of users go to the Test Set
    random_state=SEED,       # Ensures we get the exact same users every time
    stratify=stratify_labels # Ensures fair distribution of BMI categories
)

# After this, 'train_users' is a list of ID numbers. We will filter the main dataframe next.
print(f"Split Result: {len(train_users)} Train Users, {len(test_users)} Test Users")

Split Result: 420 Train Users, 180 Test Users


In [10]:
# Give me all rows where the User_ID is inside the train_users list and create a new independent table with .copy().
df_train = df_progress[df_progress['User_ID'].isin(train_users)].copy()

# Give me all rows where the User_ID is inside the test_users list and create a new independent table with .copy().
df_test = df_progress[df_progress['User_ID'].isin(test_users)].copy()

# Print the number of unique users in each set to confirm the split worked.
print(f"Final Split: {len(train_users)} Users in Train | {len(test_users)} Users in Test")

Final Split: 420 Users in Train | 180 Users in Test


## Modelling

In [11]:
# Input Features (X)
# These are the columns the model will "study" to make predictions.
features = [
    # User Profile
    'Age', 'Gender_Encoded', 'Height_cm', 
    'Initial_Weight_kg', 'Initial_BMI', 'BMI_Category_x_Encoded',
    
    # User Body Composition
    'Body_Fat_Category', 'Body_Fat_Percentage_x', 
    
    # User Lifestyle & Goals 
    'Goal_Encoded', 'Workout_Frequency', 'Average_Duration_Minutes', 'level_Encoded',
    
    # User Sports Interests 
    'Badminton', 'Football', 'Basketball', 'Volleyball', 'Swim',
    
    # Time Factor
    'Week',
]

# Create Feature Matrices
# Extract only the columns listed above from our Train/Test dataframes.
X_train = df_train[features]
X_test = df_test[features]

# Define Targets (y)
# These are the "Answers" we want the model to predict.

# Numeric Targets (Regression)
targets_num = [
    'Weight_kg', 'BMI', 'Body_Fat_Percentage_y', 'Daily_Calories', 
    'Daily_Water_ml', 'Target_Protein_g', 'Target_Carbs_g', 'Target_Fat_g',
    'Limit_Sugar_g', 'Target_Fiber_g', 'Limit_Cholesterol_mg', 
    'Target_Calcium_mg'
]

# Categorical Targets (Classification)
targets_cat = ['BMI_Category_y_Encoded', 'Meal_Frequency_Encoded']

### Model and Hyperparameter Tuning

In [12]:
# Dictionary to store trained models for later use
models = {}

# List to collect evaluation results for the final report
evaluation_report = []

# Hyperparameter Grid: The "Settings" we test to find the best model.
xgb_reg_params = {
    'n_estimators': [500, 1000],      # More trees = better learning
    'learning_rate': [0.01, 0.05],    # Slower = more precise
    'max_depth': [3, 5, 7],           # Depth of decision trees
    'subsample': [0.7, 0.9],          # Use 70-90% of rows (prevents overfitting)
    'colsample_bytree': [0.7, 0.9]    # Use 70-90% of columns (prevents overfitting)
}


# Regression loop to predict numerical columns
for t in targets_num:
    y_train = df_train[t]
    y_test = df_test[t]
    
    # Initialize Model
    xgb_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1)
    
    # Hyperparameter Tuning
    # Tries 10 random combinations of settings to find the best one
    search = RandomizedSearchCV(xgb_model, xgb_reg_params, n_iter=10, cv=3, scoring='r2', verbose=0, random_state=42)
    search.fit(X_train, y_train)
    best_model = search.best_estimator_
    
    # Evaluate
    train_score = best_model.score(X_train, y_train) # R2 Score (0.0 to 1.0)
    test_score = best_model.score(X_test, y_test)
    test_mae = mean_absolute_error(y_test, best_model.predict(X_test))
    
    # Save & Log
    models[t] = {'model': best_model, 'type': 'numeric'}
    
    evaluation_report.append({
        'Target': t,
        'Type': 'Regression',
        'Train Score': f"{train_score*100:.1f}%",
        'Test Score': f"{test_score*100:.1f}%",
        'Gap': f"{(train_score-test_score)*100:.1f}%",
        'Error Metric': f"MAE: {test_mae:.2f}"
    })

# Classification loop to predict categorical columns
for t in targets_cat:
    y_train = df_train[t]
    y_test = df_test[t]
    
    # Initialize Model (Softmax for multi-class)
    xgb_clf = xgb.XGBClassifier(objective='multi:softmax', eval_metric='mlogloss', random_state=42)
    
    # Tuning (Simplified grid for classification)
    clf_params = {'max_depth': [3, 5], 'n_estimators': [200, 500], 'learning_rate': [0.05]}
    search = RandomizedSearchCV(xgb_clf, clf_params, n_iter=5, cv=3, scoring='accuracy', random_state=42)
    search.fit(X_train, y_train)
    best_model = search.best_estimator_
    
    # Evaluate
    train_acc = accuracy_score(y_train, best_model.predict(X_train))
    test_acc = accuracy_score(y_test, best_model.predict(X_test))
    
    # Save & Log
    models[t] = {'model': best_model, 'type': 'categorical'}
    
    evaluation_report.append({
        'Target': t,
        'Type': 'Classification',
        'Train Score': f"{train_acc*100:.1f}%",
        'Test Score': f"{test_acc*100:.1f}%",
        'Gap': f"{(train_acc-test_acc)*100:.1f}%",
        'Error Metric': "Accuracy"
    })

# Creating evaluation for each target
results_df = pd.DataFrame(evaluation_report)
results_df = results_df[['Target', 'Type', 'Train Score', 'Test Score', 'Gap', 'Error Metric']]

print("\n=== Final Evaluation Report ===")
results_df

c:\Users\crisv\anaconda3\envs\ML\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 4 is smaller than n_iter=5. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
c:\Users\crisv\anaconda3\envs\ML\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 4 is smaller than n_iter=5. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(



=== Final Evaluation Report ===


,Target,Type,Train Score,Test Score,Gap,Error Metric
0,Weight_kg,Regression,100.0%,99.8%,0.1%,MAE: 0.68
1,BMI,Regression,100.0%,99.8%,0.2%,MAE: 0.26
2,Body_Fat_Percentage_y,Regression,100.0%,99.9%,0.1%,MAE: 0.38
3,Daily_Calories,Regression,99.8%,97.1%,2.7%,MAE: 47.80
4,Daily_Water_ml,Regression,100.0%,99.7%,0.2%,MAE: 31.86
5,Target_Protein_g,Regression,99.8%,98.6%,1.3%,MAE: 3.26
6,Target_Carbs_g,Regression,99.9%,99.4%,0.6%,MAE: 4.15
7,Target_Fat_g,Regression,99.9%,98.7%,1.1%,MAE: 1.19
8,Limit_Sugar_g,Regression,99.7%,96.9%,2.8%,MAE: 1.16
9,Target_Fiber_g,Regression,99.6%,96.8%,2.8%,MAE: 0.70


### Model Dump

In [13]:
# We package everything into a single dictionary "box".
# This ensures that when we load the model later, we also have the
# exact encoders and feature list used to train it.
data_progress = {
    'models_dict': models,     # The Trained AI Models (The Brain)
    'encoders': le_dict,       # The Translators (Text -> Numbers)
    'features': features       # The Map (Order of inputs is CRITICAL!)
}

# Define the save path.
# 'wb' means "Write Binary" (required for saving models).
save_path = '../../models/model_progress.pickle'

with open(save_path, 'wb') as f:
    pickle.dump(data_progress, f)

print("\n 'model_progress.pickle' saved.")


 'model_progress.pickle' saved.
